# 🌊 DRISHTI-SSS: Google Colab Model Training Pipeline
### Smart India Hackathon 2026 — Problem Statement 26057 (MoES / NIOT)
**Dataset:** `rehan9599/drishti-sss` (5,205 tiles of Side-Scan Sonar Debris & Anomalies)

---

### Step 1: Check GPU & Install Dependencies

In [ ]:
!nvidia-smi
!pip install -q ultralytics huggingface_hub onnx onnxruntime pyyaml

: 

### Step 2: Download `rehan9599/drishti-sss` Dataset from Hugging Face into Cloud RAM

In [ ]:
import os
import yaml
from huggingface_hub import snapshot_download

dataset_dir = "/content/drishti_dataset"
print("Pulling dataset from Hugging Face...")
snapshot_download(
    repo_id="rehan9599/drishti-sss",
    repo_type="dataset",
    local_dir=dataset_dir,
    ignore_patterns=[".git*", "*.parquet", "*.md"]
)
print("✓ Download complete!")

### Step 3: Configure Dataset YAML for Debris Classes (0-indexed)

In [ ]:
data_yaml_path = os.path.join(dataset_dir, "drishti_colab.yaml")
dataset_cfg = {
    'path': dataset_dir,
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'names': {
        0: 'crab_pot',
        1: 'submarine_pipeline',
        2: 'shipwreck',
        3: 'ghost_net',
        4: 'mine_cylinder'
    }
}

with open(data_yaml_path, 'w') as f:
    yaml.dump(dataset_cfg, f, default_flow_style=False)
print(f"✓ Created YAML config at {data_yaml_path}")

### Step 4: Train YOLOv8s on Cloud GPU with Acoustic Augmentations

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

results = model.train(
    data=data_yaml_path,
    epochs=80,
    imgsz=640,
    batch=16,
    mosaic=1.0,
    mixup=0.15,
    fliplr=0.5,
    flipud=0.0,
    degrees=15.0,
    scale=0.3,
    hsv_v=0.4,
    device=0,
    name="drishti_yolov8s_run"
)

### Step 5: Export to ONNX & Download Weights

In [ ]:
# Export ONNX format for edge deployment
onnx_path = model.export(format="onnx", dynamic=True, simplify=True)
print(f"✓ Exported ONNX to {onnx_path}")

# Download weights directly to your browser
from google.colab import files
best_pt = os.path.join(model.trainer.save_dir, "weights", "best.pt")
print("Downloading best.pt...")
files.download(best_pt)
print("Downloading ONNX model...")
files.download(onnx_path)
print("Done! Place these files into your local 'models/weights/' directory.")